# 02 · Learn a positive Lyapunov candidate

**Goal:** train the actual Gram-feature model used in `experiments/van_der_pol/EVT/commons.py`,
then evaluate it on fresh data. This notebook is independent of notebook 01 and
uses 300 small CPU steps. It writes only `tutorials/outputs/tutorial_candidate.pt`.

The research repository has several baselines. Here we focus on the EVT pipeline's
trainable feature/Gram model, not the separate tanh network in `NLF/`.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import torch
import matplotlib.pyplot as plt

# Work from the repository root or anywhere inside it.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "tutorials/_support.py").is_file())
if str(ROOT / "tutorials") not in sys.path:
    sys.path.insert(0, str(ROOT / "tutorials"))
from _support import (setup, quadratic_matrix, warm_start, lie_derivative,
                      grid, sample_shell, VanDerPol)
rng = setup(42)
physics = VanDerPol(mu=1.0)
plt.rcParams.update({"figure.figsize": (6, 4), "axes.grid": False})
print("CPU tutorial | PyTorch", torch.__version__)

## 1. Positivity by construction

The ten features are
$$z(x)=[x_1,x_2,x_1^2,x_1x_2,x_2^2,\log\cosh x_1,\log\cosh x_2,
 x_1\tanh x_2,x_2\tanh x_1,x_1\tanh x_1\,x_2].$$
We learn $L$ and define
$$Q=LL^T+10^{-4}I,\qquad V(x)=z(x)^TQz(x).$$
Every feature vanishes at zero, and the first two features contain the state.
Thus $V(0)=0$ and $V(x)\ge10^{-4}\|x\|^2$ mathematically. This also makes
sublevels bounded. **Decrease is not guaranteed by this architecture.**

We initialize the leading block from the linear Lyapunov equation in notebook 01.
The remaining $10^{-4}I$ contribution makes this close to, but not exactly, its
quadratic candidate. Inspect `_support.warm_start` for the short initialization.

In [ ]:
from copy import deepcopy
from experiments.van_der_pol.EVT.commons import construct_ode_monomials

model = warm_start()
baseline = deepcopy(model)
print("Feature count:", construct_ode_monomials(torch.zeros(1, 2)).shape[1])
assert model(torch.zeros(1, 2)).item() == 0.0
assert torch.linalg.eigvalsh(model.get_Q()).min() > 0
print("Trainable parameters:", sum(p.numel() for p in model.parameters()))

## 2. Turn the derivative condition into a loss

For random states in a small training box, penalize
$\mathrm{ReLU}(\dot V+0.1V)$. A shape term
$0.5\,(V-\|x\|^2)^2$ discourages simply shrinking the candidate's scale.
This mirrors the terms in the research trainer, on a smaller demonstration domain.
The derivative target vanishes at the origin, as it should.

`create_graph=True` is essential: optimizing $L$ through the derivative of $V$
requires differentiating the gradient calculation itself. Training samples change
at every step. There is no claim that this finite optimization enforces the
inequality at all points, or that every diagnostic improves monotonically.

In [ ]:
EPOCHS = 300
BATCH_SIZE = 256
TRAIN_LIMIT = 0.9
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
history = []
for epoch in range(EPOCHS):
    points = (torch.rand(BATCH_SIZE, 2) * 2 - 1) * TRAIN_LIMIT
    values, derivatives, _ = lie_derivative(model, points, create_graph=True)
    decrease_loss = torch.relu(derivatives + 0.1 * values).mean()
    shape_loss = ((values - points.square().sum(dim=1))**2).mean()
    loss = decrease_loss + 0.5 * shape_loss
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    history.append([loss.item(), decrease_loss.item(), shape_loss.item()])
assert np.isfinite(history).all()
fig, ax = plt.subplots()
ax.semilogy(np.maximum(history, 1e-10))
ax.legend(["Total", "Decrease", "Shape"])
ax.set(xlabel="Training step", ylabel="Loss", title="Stochastic training diagnostics")
plt.show()

## 3. Check states the optimizer did not see

Report the fraction with positive derivative and the largest derivative on an
independent sample. Compare both the training box and a larger box. A positive
value directly exhibits a failed decrease condition. Zero observed violations
does not establish that the true violation probability is zero.

In [ ]:
generator = torch.Generator().manual_seed(2026)
for limit in [TRAIN_LIMIT, 2.0]:
    held_out = (torch.rand(5000, 2, generator=generator) * 2 - 1) * limit
    for name, candidate in [("Warm start", baseline), ("Trained", model)]:
        values, derivatives, _ = lie_derivative(candidate, held_out)
        assert torch.isfinite(values).all() and torch.isfinite(derivatives).all()
        assert (values.detach() >= 0).all()
        print(f"{name:10s} | box +/-{limit:.1f} | positive derivative: "
              f"{(derivatives > 0).float().mean().item():.2%} | "
              f"sample max: {derivatives.max().item():.4f}")

In [ ]:
X, Y, points = grid()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, candidate, title in zip(axes, [baseline, model], ["Warm start", "Trained"]):
    values, derivatives, _ = lie_derivative(candidate, points)
    V = values.detach().numpy().reshape(X.shape)
    D = derivatives.detach().numpy().reshape(X.shape)
    ax.contourf(X, Y, D > 0, levels=[-0.5, 0.5, 1.5], colors=["#e3f1ed", "#f7b6ae"])
    contours = ax.contour(X, Y, V, levels=[0.1, 0.3, 0.6, 1.0, 2.0], colors="black")
    ax.clabel(contours, fontsize=8)
    ax.add_patch(plt.Rectangle((-TRAIN_LIMIT, -TRAIN_LIMIT), 2*TRAIN_LIMIT, 2*TRAIN_LIMIT,
                              fill=False, linestyle="--", edgecolor="blue"))
    ax.set(xlabel="$x_1$", ylabel="$x_2$", title=title + " | red: positive derivative")
plt.tight_layout()
plt.show()

## 4. Save a teaching checkpoint

The checkpoint stores configuration and weights. It is separate from every
research artifact. Notebook 03 can load it, or run independently with its own
warm-started candidate. The two modes assess different candidates: record which
one you use when comparing results.

In [ ]:
output = ROOT / "tutorials/outputs"
output.mkdir(parents=True, exist_ok=True)
checkpoint = output / "tutorial_candidate.pt"
torch.save({"state_dict": model.state_dict(), "seed": 42, "epochs": EPOCHS,
            "train_limit": TRAIN_LIMIT, "mu": 1.0, "feature_dim": 10}, checkpoint)
print("Saved:", checkpoint.relative_to(ROOT))

## Try it

1. Remove the shape term. Track candidate scale as well as the derivative loss.
2. Change the training box to `2.0`. Why is global decrease an unreasonable target
   for a system whose basin is bounded?
3. Compare sublevel geometry rather than comparing raw rho values across models:
   rescaling V changes rho without changing a region's shape.

**Next:** [03 — Assess a candidate with EVT](03_evt_assessment.ipynb).